# Stage 5 — NER Evaluation via Manual Annotation

**Goal:** Estimate the precision and recall of CamemBERT-NER on the manifesto corpus.

**Procedure:**
- Sample 150 manifesto excerpts (first 2000 chars, stratified by gender)
- For each excerpt: the model's predicted person names are shown
- Mark each prediction as correct (TP) or wrong (FP)
- Mark any person names the model **missed** (FN)
- From these, precision = TP/(TP+FP) and recall = TP/(TP+FN) are computed.

## 0. Setup & Sample

In [1]:
import pandas as pd
import json
from pathlib import Path
from transformers import pipeline

PROJECT_ROOT = Path("..")

# Load manifestos
df = pd.read_csv(PROJECT_ROOT / "data" / "manifestos_with_metadata.csv", low_memory=False)

# Sample 25 female + 25 male manifestos (stratified)
sample_f = df[df["titulaire-sexe"] == "femme"].sample(25, random_state=42)
sample_m = df[df["titulaire-sexe"] == "homme"].sample(25, random_state=42)
sample = pd.concat([sample_f, sample_m]).reset_index(drop=True)

print(f"Sample size: {len(sample)} manifestos")
print(sample["titulaire-sexe"].value_counts())

# Load NER model
ner = pipeline(
    "ner",
    model="Jean-Baptiste/camembert-ner",
    aggregation_strategy="simple",
    device="mps"  # change to 0 for GPU, -1 for CPU
)
print("Model loaded.")

Sample size: 50 manifestos
titulaire-sexe
femme    25
homme    25
Name: count, dtype: int64


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

CamembertForTokenClassification LOAD REPORT from: Jean-Baptiste/camembert-ner
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded.


## 1. Run NER on sample & prepare annotation file

We run NER on the 50 sampled manifestos and save the predictions to a CSV for annotation.

In [5]:
rows = []
for _, row in sample.iterrows():
    text = str(row["text"])[:2000]
    entities = ner(text)
    persons = [e for e in entities if e["entity_group"] == "PER" and e["score"] > 0.7]

    for p in persons:
        rows.append({
            "candidate_id":   row["candidate_id"],
            "candidate_sex":  row["titulaire-sexe"],
            "text_excerpt":   text,              # full 2000 chars for annotation context
            "predicted_name": p["word"],
            "score":          round(p["score"], 3),
            "correct":        "",                # TO FILL: 1=correct, 0=wrong
            "missed_names":   "",                # TO FILL: names the model missed (comma-separated)
        })

df_annot = pd.DataFrame(rows)
annot_path = PROJECT_ROOT / "data" / "annotation_sample.csv"
df_annot.to_csv(annot_path, index=False)
print(f"Saved {len(df_annot)} predictions to {annot_path}")
print(f"\nSample of predictions:")
print(df_annot[["candidate_id", "candidate_sex", "predicted_name", "score"]].head(20).to_string())

Saved 154 predictions to ../data/annotation_sample.csv

Sample of predictions:
                      candidate_id candidate_sex            predicted_name  score
0   EL198_L_1993_03_092_06_1_PF_01         femme             Nadine GARCIA  0.998
1   EL198_L_1993_03_092_06_1_PF_01         femme             Thierry DANIS  0.997
2   EL198_L_1993_03_092_06_1_PF_01         femme                  . Rocard  0.905
3   EL106_L_1978_03_075_24_1_PF_04         femme              Mademoiselle  0.801
4   EL106_L_1978_03_075_24_1_PF_04         femme                  Monsieur  0.719
5   EL198_L_1993_03_092_03_1_PF_06         femme         Catherine BRIGAND  0.998
6   EL198_L_1993_03_092_03_1_PF_06         femme                  GAMBAZZA  0.988
7   EL198_L_1993_03_092_03_1_PF_06         femme            Jacques CHOLET  0.995
8   EL198_L_1993_03_092_03_1_PF_06         femme             Brice LALONDE  0.999
9   EL137_L_1981_06_088_04_1_PF_01         femme  NEUFCHATEAU Maria ROUYER  0.770
10  EL107_L_1978_03

## 2. Annotation instructions

Open `data/annotation_sample.csv` in Excel or Numbers.

For each row (= one predicted name):
- Read the `text_excerpt` for context
- Fill `correct` column: **1** if the predicted name is a real person name, **0** if it's wrong (title, noise, not a person)
- In the `missed_names` column of the **first row for each candidate_id**: write any person names you see in the excerpt that the model did NOT predict (comma-separated)

Save the file when done and run the evaluation cell below.

## 3. Compute precision & recall

In [2]:
# Run after completing annotation in the CSV
df_annot = pd.read_csv(PROJECT_ROOT / "data" / "annotation_sample_filled_out.csv", skiprows=1)

TP = df_annot["correct"].sum()
FP = (df_annot["correct"] == 0).sum()

# Count missed names (FN)
missed = df_annot["missed_names"].dropna().str.split(",")
FN = missed.apply(lambda x: len([n for n in x if n.strip()])).sum()

precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall    = TP / (TP + FN) if (TP + FN) > 0 else 0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print(f"True Positives  (TP): {TP}")
print(f"False Positives (FP): {FP}")
print(f"False Negatives (FN): {FN}")
print(f"\nPrecision: {precision:.3f}")
print(f"Recall:    {recall:.3f}")
print(f"F1 score:  {f1:.3f}")

# By gender
for sex in ["femme", "homme"]:
    sub = df_annot[df_annot["candidate_sex"] == sex]
    tp = sub["correct"].sum()
    fp = (sub["correct"] == 0).sum()
    p = tp / (tp + fp) if (tp + fp) > 0 else 0
    print(f"\nPrecision ({sex}): {p:.3f}")

True Positives  (TP): 132
False Positives (FP): 22
False Negatives (FN): 3

Precision: 0.857
Recall:    0.978
F1 score:  0.913

Precision (femme): 0.812

Precision (homme): 0.913
